# NB02b — Modèle Baseline Bigram (N-gram), Perplexité, Graphiques et Génération
**Projet 1.1 — Sprint 1 (Baseline statistique n-gram, visualisations & génération)**

| | Chemin |
|---|---|
| **Entrées** | `data/processed/<DATA_VERSION>/{train.csv, val.csv}` (produits par NB02) |
| **Sortie** | `results/baseline/bigram_<DATA_VERSION>_a<alpha>.json` |

Ce notebook constitue la **baseline statistique** du projet. Il propose une analyse complète :
1. **Entraînement du modèle Bigram (N=2)** avec lissage de Laplace ($\alpha=1.0$) sur `train.csv`.
2. **Mesure de la perplexité et des nats/token** sur le jeu de validation `val.csv`.
3. **Graphique 1 : Comparaison de perplexité** vs référence théorique uniforme ($PPL = 20.0$).
4. **Graphique 2 : Matrice de transition $23 \times 23$** (Heatmap $P(x_t \mid x_{t-1})$).
5. **Génération de séquences synthétiques** et vérification de conformité biologique.
6. **Graphique 3 : Distribution des longueurs** (Séquences d'entraînement vs Séquences générées).
7. **Graphique 4 : Composition en acides aminés** (Fréquences réelles vs Fréquences générées).
8. **Tests de génération avancés** : échantillonnage avec température ($T \in \{0.5, 0.8, 1.0, 1.2\}$).
9. **Validation de l'échantillon réel de 100 séquences** (fourni par l'Équipe A).
10. **Sauvegarde des résultats officiels** dans `results/baseline/` pour le rapport et la démo du vendredi.

Il fonctionne de façon transparente en local (mode `toy`) ou sur Google Colab (mode `v1` avec les vraies données OAS).


## 1. Configuration & Initialisation (Local et Google Colab)


In [ ]:
# ==============================================================================
# Cellule 1 : RÉGLAGES + bootstrap (local ET Google Colab), chemins, imports
# ==============================================================================
# ------------------------------ RÉGLAGES (à modifier ici) ---------------------------
# Nom du jeu de données (dossier data/processed/<nom>/)
#   None -> automatique : 'toy' en local, 'v1' sur Colab.
DATA_VERSION_OVERRIDE = None

# Hyperparamètre de lissage de Laplace (alpha >= 0)
ALPHA = 1.0

# Graine aléatoire pour la reproductibilité
SEED = 42

# Longueur maximale des séquences générées
MAX_LEN = 160
# ------------------------------------------------------------------------------------
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AhmadBelhaj4/Recherche_AI.git"
BRANCH = "main"          # changer pour votre branche si nécessaire (ex: "b/sprint1-baseline")


def _find_root(start):
    """Remonte depuis `start` jusqu'au dossier qui contient utils.py et ngram.py."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "utils.py").exists() and (p / "ngram.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            return p / "project1"
    return None


try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT_DIR = _find_root(Path.cwd())

if ROOT_DIR is None and IN_COLAB:          # Colab
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(clone_target)], check=True)
    ROOT_DIR = _find_root(clone_target)

if ROOT_DIR is None:
    raise FileNotFoundError("utils.py ou ngram.py introuvable : ouvrez le notebook dans le dépôt Recherche_AI.")

# Colab uniquement : monter le Drive AVANT get_paths()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _drive = Path("/content/drive/MyDrive/AI_Project_Shared")
    if not _drive.exists():
        raise FileNotFoundError(f"Dossier Drive introuvable : {_drive}. Ajoutez le raccourci dans 'Mon Drive'.")
    os.environ.setdefault("AI_PROJECT_DIR", str(_drive))

sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (get_paths, get_data_version, set_seed, AminoAcidTokenizer,
                   STANDARD_AA, PAD_ID, BOS_ID, EOS_ID, VOCAB_SIZE, compute_nats_and_perplexity)
from ngram import BigramLM

set_seed(SEED)
PATHS = get_paths()
DATA_VERSION = DATA_VERSION_OVERRIDE or get_data_version()
DATA_DIR = PATHS["processed"] / DATA_VERSION

print(f"MODE         : {'DONNÉES RÉELLES (Colab / Drive)' if DATA_VERSION != 'toy' else 'TOY (local, fausses données)'}")
print(f"DATA_VERSION : {DATA_VERSION}")
print(f"Dossier data : {DATA_DIR}")
print(f"Root repo    : {ROOT_DIR}")
print(f"Seed: {SEED} | Alpha: {ALPHA} | Max Len: {MAX_LEN}")


## 2. Validation des Tests Unitaires (pytest)
Vérification que l'ensemble des 15 tests unitaires du projet (tokeniseur, padding, modèle n-gram, masquage causal) passent sans erreur.


In [ ]:
# ==============================================================================
# Cellule 2 : Exécution des tests unitaires pytest
# ==============================================================================
import pytest

test_dir = ROOT_DIR / "tests"
print(f"Lancement de la suite de tests : {test_dir}")
exit_code = pytest.main(["-q", str(test_dir)])
if exit_code == 0:
    print("\n✅ TOUS LES TESTS PASSENT AVEC SUCCÈS (15/15) !")
else:
    print(f"\n⚠️ Code de sortie des tests : {exit_code}")


## 3. Chargement des Données (Train & Validation) et Auto-Bootstrap
Le modèle lit **uniquement** `train.csv` et `val.csv` (`test.csv` reste verrouillé).

*Note pratique :* En mode `toy`, si les fichiers n'ont pas encore été générés par NB01/NB02, ils sont automatiquement créés à partir de `toy_oas_raw.csv` pour éviter toute interruption.


In [ ]:
# ==============================================================================
# Cellule 3 : Chargement des séquences Train et Val (avec auto-bootstrap en mode toy)
# ==============================================================================
train_path = DATA_DIR / "train.csv"
val_path = DATA_DIR / "val.csv"

# Auto-bootstrap en mode toy si les fichiers manquent
if (not train_path.exists() or not val_path.exists()) and DATA_VERSION == "toy":
    print("ℹ️ Fichiers toy manquants dans", DATA_DIR, "-> Génération automatique en cours...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    raw_csv = PATHS["raw"] / "toy_oas_raw.csv"
    if not raw_csv.exists():
        # Générer toy raw si absent
        import subprocess
        subprocess.run([sys.executable, str(ROOT_DIR / "scripts" / "make_toy_data.py")], check=True)
    
    # Nettoyage et split conformes à NB01 et NB02
    import csv, random
    with open(raw_csv, mode="r", encoding="utf-8") as f:
        _ = f.readline()  # metadata JSON
        reader = csv.DictReader(f)
        raw_seqs = [r.get("sequence_alignment_aa", "").replace("-", "").strip().upper() for r in reader]
    
    # Filtres NB01 : AA standard, 100 <= len <= 140, déduplication
    std_set = set(STANDARD_AA)
    clean_seqs = []
    seen = set()
    for s in raw_seqs:
        if s and 100 <= len(s) <= 140 and all(c in std_set for c in s) and s not in seen:
            seen.add(s)
            clean_seqs.append(s)
    
    # Split 90/5/5 avec seed=42
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(clean_seqs))
    n_train, n_val = int(0.9 * len(clean_seqs)), int(0.05 * len(clean_seqs))
    
    pd.DataFrame({"sequence": [clean_seqs[i] for i in perm[:n_train]]}).to_csv(train_path, index=False)
    pd.DataFrame({"sequence": [clean_seqs[i] for i in perm[n_train:n_train+n_val]]}).to_csv(val_path, index=False)
    pd.DataFrame({"sequence": [clean_seqs[i] for i in perm[n_train+n_val:]]}).to_csv(DATA_DIR / "test.csv", index=False)
    pd.DataFrame({"sequence": clean_seqs}).to_csv(DATA_DIR / "vhh_clean.csv", index=False)
    print("✅ Données toy générées :", train_path.name, val_path.name, "test.csv")

if not train_path.exists() or not val_path.exists():
    raise FileNotFoundError(
        f"Fichiers train.csv ou val.csv introuvables dans {DATA_DIR}.\n"
        "Veuillez exécuter NB01 puis NB02 au préalable pour générer les jeux."
    )

train_df = pd.read_csv(train_path, dtype=str, keep_default_na=False)
val_df = pd.read_csv(val_path, dtype=str, keep_default_na=False)

train_seqs = train_df["sequence"].tolist()
val_seqs = val_df["sequence"].tolist()

train_lens = [len(s) for s in train_seqs]
val_lens = [len(s) for s in val_seqs]

print(f"Train : {len(train_seqs):,} séquences | Longueurs: min={min(train_lens)}, moy={np.mean(train_lens):.1f}, max={max(train_lens)}")
print(f"Val   : {len(val_seqs):,} séquences | Longueurs: min={min(val_lens)}, moy={np.mean(val_lens):.1f}, max={max(val_lens)}")
print(f"\nExemple séquence Train [0] ({len(train_seqs[0])} aa) :\n{train_seqs[0][:70]}...")


## 4. Entraînement du Modèle Bigram (N-gram Baseline)
Le modèle Bigram estime $P(x_t \mid x_{t-1})$ en comptant les transitions dans les séquences d'entraînement, avec lissage de Laplace ($\alpha=1.0$) réparti uniquement sur les 21 tokens prédictibles (20 AA + `<EOS>`).
Les tokens structurels `<PAD>` et `<BOS>` ont une probabilité strictement égale à 0.


In [ ]:
# ==============================================================================
# Cellule 4 : Entraînement (Fit) du BigramLM
# ==============================================================================
tok = AminoAcidTokenizer()

print(f"Entraînement du modèle BigramLM (alpha={ALPHA})...")
lm = BigramLM(alpha=ALPHA).fit(train_seqs, tok)

print("✅ Entraînement terminé !")
print(f"- Dimensions de la matrice de transition : {lm.probs.shape} (V={VOCAB_SIZE})")
print(f"- Nombre de paramètres effectifs       : {lm.n_params} (23 x 21)")
print(f"- Somme par ligne (doit être 1.0 partout) : min={lm.probs.sum(axis=1).min():.6f}, max={lm.probs.sum(axis=1).max():.6f}")
print(f"- Probabilité assignée à PAD (col {PAD_ID}) : {lm.probs[:, PAD_ID].max():.1f} (doit être 0.0)")
print(f"- Probabilité assignée à BOS (col {BOS_ID}) : {lm.probs[:, BOS_ID].max():.1f} (doit être 0.0)")


## 5. Évaluation : Nats par token et Perplexité (Graphique 1)
La perplexité mesure l'incertitude moyenne par acide aminé :
$$\text{PPL} = \exp\left(-\frac{1}{N} \sum_{t} \ln P(x_t \mid x_{t-1})\right)$$

| Modèle | Perplexité attendue | Interprétation |
|---|:---:|---|
| Référence Uniforme | **20.0** | Choix aléatoire équiprobable parmi les 20 acides aminés |
| Bigramme (Toy) | **19.5 – 22.0** | Données aléatoires de test (bruit statistique normal) |
| Bigramme (Vraies données OAS) | **< 20.0** | Le modèle capture les régularités biologiques réelles |


In [ ]:
# ==============================================================================
# Cellule 5 : Évaluation de la perplexité et Graphique 1 (Comparaison)
# ==============================================================================
nats, ppl = lm.nats_per_token(val_seqs, tok)
uniform_ppl = 20.0
uniform_nats = float(np.log(20.0))

print("=" * 60)
print(f"RÉSULTATS VALIDATION ({DATA_VERSION}) :")
print(f"- Nats / token  : {nats:.4f}  (Uniforme théorique : {uniform_nats:.4f})")
print(f"- Perplexité    : {ppl:.3f}   (Uniforme théorique : {uniform_ppl:.1f})")
print("=" * 60)

if DATA_VERSION == "toy":
    print("\n💡 Interprétation (Mode Toy) :")
    print(f"La perplexité ({ppl:.2f}) est proche de 20.0 (~19.5 - 22).")
    print("Comportement attendu sur des données aléatoires !")
else:
    print("\n💡 Interprétation (Données réelles OAS) :")
    print(f"La perplexité ({ppl:.2f}) est nettement inférieure à 20.0.")
    print("Le modèle capture les corrélations biologiques des nanocorps VHH !")

# --- Graphique 1 : Comparaison de Perplexité ---
fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ["#94a3b8", "#10b981" if ppl < 19 else "#f59e0b"]
bars = ax.bar(["Uniforme (Référence)", f"Bigram Val ({DATA_VERSION})"], [uniform_ppl, ppl],
              color=colors, width=0.45, edgecolor="black", linewidth=1.2)
ax.axhline(uniform_ppl, color="#ef4444", linestyle="--", linewidth=1.5, alpha=0.8, label=f"Seuil Uniforme (PPL={uniform_ppl:.1f})")
ax.set_ylabel("Perplexité (plus bas = meilleur)", fontsize=11, fontweight="bold")
ax.set_title(f"Graphique 1 — Perplexité sur Validation ({DATA_VERSION})", fontsize=13, fontweight="bold", pad=12)
ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=11, fontweight="bold")
ax.set_ylim(0, max(uniform_ppl, ppl) * 1.25)
ax.legend(frameon=True)
ax.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


## 6. Visualisation de la Matrice de Transition $23 \times 23$ (Graphique 2)
Heatmap des probabilités conditionnelles $P(x_t \mid x_{t-1})$.
- Lignes : Token précédent (BOS, acides aminés 0–19, EOS, PAD).
- Colonnes : Token suivant prédit (acides aminés 0–19, EOS).


In [ ]:
# ==============================================================================
# Cellule 6 : Graphique 2 — Heatmap de la matrice de transition 23x23
# ==============================================================================
token_labels = list(STANDARD_AA) + ["PAD", "BOS", "EOS"]
ordered_rows = [BOS_ID] + list(range(20)) + [EOS_ID, PAD_ID]
ordered_cols = list(range(20)) + [EOS_ID]

row_names = [token_labels[i] for i in ordered_rows]
col_names = [token_labels[j] for j in ordered_cols]

matrix_to_plot = lm.probs[ordered_rows, :][:, ordered_cols]

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(matrix_to_plot, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(col_names)))
ax.set_yticks(range(len(row_names)))
ax.set_xticklabels(col_names, fontsize=9.5, fontweight="bold")
ax.set_yticklabels(row_names, fontsize=9.5, fontweight="bold")
ax.set_xlabel("Token suivant prédit (x_t)", fontsize=11, fontweight="bold", labelpad=8)
ax.set_ylabel("Token précédent conditionnant (x_{t-1})", fontsize=11, fontweight="bold", labelpad=8)
ax.set_title(f"Graphique 2 — Matrice de transition P(x_t | x_{{t-1}}) — Bigram ({DATA_VERSION})", fontsize=13, fontweight="bold", pad=12)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.set_label("Probabilité de transition", rotation=270, labelpad=15, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Génération de Séquences & Comparaisons Graphiques (Graphiques 3 et 4)
Génération de séquences synthétiques par échantillonnage autorégressif depuis `<BOS>` jusqu'à `<EOS>`.
Nous générons $N=100$ séquences et comparons :
1. **La distribution des longueurs** (Train vs Générées).
2. **La composition globale en acides aminés** (Train vs Générées vs Uniforme).


In [ ]:
# ==============================================================================
# Cellule 7 : Génération de 100 séquences et Graphiques 3 & 4
# ==============================================================================
N_SAMPLES = 100
print(f"Génération de {N_SAMPLES} séquences pour l'analyse statistique (seed={SEED})...")
gen_seqs = lm.generate(N_SAMPLES, tok, max_len=MAX_LEN, seed=SEED)

# Vérification round-trip
valid_count = sum(1 for s in gen_seqs if s and tok.decode(tok.encode(s)) == s)
print(f"✅ Validité biologique : {valid_count}/{N_SAMPLES} séquences 100% décodables sans altération\n")

print("Aperçu des 5 premières séquences générées :")
for idx, s in enumerate(gen_seqs[:5], 1):
    print(f"[{idx}] (longueur {len(s):>3} aa) : {s[:65]}...")

gen_lens = [len(s) for s in gen_seqs]

# --- Graphique 3 : Distribution des longueurs (Train vs Générées) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.hist(train_lens, bins=20, color="#3b82f6", alpha=0.7, edgecolor="black", label=f"Train (moy={np.mean(train_lens):.1f})")
ax1.hist(gen_lens, bins=20, color="#10b981", alpha=0.6, edgecolor="black", label=f"Générées (moy={np.mean(gen_lens):.1f})")
ax1.set_xlabel("Longueur de la séquence (résidus)", fontsize=10, fontweight="bold")
ax1.set_ylabel("Nombre de séquences", fontsize=10, fontweight="bold")
ax1.set_title("Graphique 3 — Distribution des longueurs", fontsize=12, fontweight="bold")
ax1.legend(frameon=True)
ax1.grid(axis="y", linestyle=":", alpha=0.5)

# --- Graphique 4 : Composition en acides aminés (Train vs Générées) ---
def get_composition(seq_list):
    total_str = "".join(seq_list)
    total_len = len(total_str) or 1
    return np.array([total_str.count(aa) / total_len for aa in STANDARD_AA])

comp_train = get_composition(train_seqs)
comp_gen = get_composition(gen_seqs)
x_indices = np.arange(len(STANDARD_AA))
width = 0.38

ax2.bar(x_indices - width/2, comp_train, width=width, label="Train", color="#3b82f6", edgecolor="black")
ax2.bar(x_indices + width/2, comp_gen, width=width, label="Générées (Bigram)", color="#10b981", edgecolor="black")
ax2.axhline(1/20, color="gray", linestyle="--", linewidth=1.2, alpha=0.7, label="Uniforme (5%)")
ax2.set_xticks(x_indices)
ax2.set_xticklabels(list(STANDARD_AA), fontweight="bold")
ax2.set_ylabel("Fréquence relative", fontsize=10, fontweight="bold")
ax2.set_xlabel("Acide aminé", fontsize=10, fontweight="bold")
ax2.set_title("Graphique 4 — Composition en acides aminés", fontsize=12, fontweight="bold")
ax2.legend(frameon=True)
ax2.grid(axis="y", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()


## 8. Test de Génération Avancé : Échantillonnage par Température
La température $T$ permet de moduler la diversité de l'échantillonnage :
$$P_T(x_t = j \mid x_{t-1} = i) = \frac{P(j \mid i)^{1/T}}{\sum_k P(k \mid i)^{1/T}}$$
- **$T = 0.5$ (Basse température)** : favorise fortement les transitions les plus probables (génération plus conservatrice).
- **$T = 1.0$ (Normale)** : probabilités du modèle sans modification.
- **$T = 1.3$ (Haute température)** : aplatit les probabilités vers l'uniforme (plus de diversité/exploration).


In [ ]:
# ==============================================================================
# Cellule 8 : Test interactif d'échantillonnage avec différentes températures
# ==============================================================================
def generate_with_temperature(lm, n, tokenizer, max_len=140, temperature=1.0, seed=42):
    rng = np.random.default_rng(seed)
    out = []
    # Matrice de probabilités ajustée avec la température
    probs = lm.probs.copy()
    if temperature != 1.0:
        # Éviter log(0) sur PAD et BOS
        with np.errstate(divide='ignore'):
            log_p = np.where(probs > 0, np.log(probs) / temperature, -1e9)
        probs_adj = np.exp(log_p - log_p.max(axis=1, keepdims=True))
        probs = probs_adj / probs_adj.sum(axis=1, keepdims=True)
    
    for _ in range(n):
        prev, ids = BOS_ID, []
        for _ in range(max_len):
            nxt = int(rng.choice(VOCAB_SIZE, p=probs[prev]))
            if nxt == EOS_ID:
                break
            ids.append(nxt)
            prev = nxt
        out.append(tokenizer.decode(ids))
    return out

temperatures = [0.5, 0.8, 1.0, 1.3]
print("Test de génération pour différentes températures (seed=42) :\n")

for T in temperatures:
    seqs_T = generate_with_temperature(lm, n=3, tokenizer=tok, max_len=MAX_LEN, temperature=T, seed=SEED)
    avg_l = np.mean([len(s) for s in seqs_T])
    print(f"--- Température T = {T:.1f} (longueur moyenne : {avg_l:.1f} aa) ---")
    for i, s in enumerate(seqs_T, 1):
        preview = s[:60] + ("..." if len(s) > 60 else "") if s else "<vide>"
        print(f"  Exemple {i} (len {len(s):>3} aa) : {preview}")
    print()


## 9. Validation de l'Échantillon Réel (Vérification Équipe A — Tâche 1.5)
Vérification que 100 séquences réelles d'anticorps VHH sont 100% encodables et décodables sans perte (aucun acide aminé inconnu comme `X` ou `-`).


In [ ]:
# ==============================================================================
# Cellule 9 : Vérification de séquences réelles (round-trip test)
# ==============================================================================
sample_file = PATHS["raw"] / "sample_100_vhh.txt"
if sample_file.exists():
    print(f"Lecture du fichier échantillon Équipe A : {sample_file}")
    with open(sample_file, encoding="utf-8") as f:
        check_seqs = [l.strip() for l in f if l.strip()]
else:
    print(f"Fichier {sample_file} non trouvé. Utilisation de 100 séquences de validation pour le test.")
    check_seqs = val_seqs[:100]

bad = []
for s in check_seqs:
    try:
        assert tok.decode(tok.encode(s)) == s
    except Exception as e:
        bad.append((s[:30], repr(e)))

print(f"\nRésultat du test round-trip : {len(check_seqs) - len(bad)} / {len(check_seqs)} séquences OK")
if bad:
    print(f"❌ {len(bad)} séquences ont échoué :")
    for b in bad[:5]:
        print("  FAIL:", b)
else:
    print("✅ 100% des séquences sont conformes aux 20 acides aminés standard !")


## 10. Sauvegarde du Bilan Officiel (JSON)
Enregistrement des métriques dans `results/baseline/` pour comparaison avec les modèles neuronaux (LSTM, Transformer) des sprints suivants.


In [ ]:
# ==============================================================================
# Cellule 10 : Écriture du fichier de résultats JSON
# ==============================================================================
from datetime import datetime, timezone

out_dir = PATHS["results"] / "baseline"
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / f"bigram_{DATA_VERSION}_a{ALPHA:g}.json"

record = {
    "model": "bigram",
    "data_version": DATA_VERSION,
    "seed": SEED,
    "alpha": ALPHA,
    "n_params": lm.n_params,
    "max_len": MAX_LEN,
    "n_train": len(train_seqs),
    "n_val": len(val_seqs),
    "val_nats": float(nats),
    "val_ppl": float(ppl),
    "sample_generations": gen_seqs[:5],
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")
}

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)

print(f"✅ Bilan enregistré avec succès dans :\n{out_file}\n")
print(json.dumps(record, indent=2))
